## Introduction

This notebook studies a simple question: when oil markets move from a normal regime into a war-driven regime, does text become more useful than it was before?

The baseline model is not price-only. It already knows about market structure, oil context, and dated event flags. The `Sent` model starts from that same base and adds text interpretation. So the comparison is not "markets versus language" in the abstract. It is "market-and-event baseline" versus "market-and-event baseline plus text meaning."

The reader's path through the notebook is deliberate:

1. Establish that `Sent` separates from `Baseline` during the war.
2. Check whether that separation appears on the kinds of days the thesis predicts.
3. Show which features become more informative in war.
4. Show that the trained model actually leans into those features.
5. Show that the gain comes from feature combination, not one isolated variable.

## Research Objective

The working hypothesis is narrow. We are not asking whether text should dominate all the time. We are asking whether text interpretation becomes incrementally more useful once oil pricing becomes more sensitive to geopolitical supply narratives.

That matters because a normal market regime and a war regime are not the same forecasting problem. Before the shock, a broad market-and-event baseline may be enough. During the shock, the market can react to statements about shipping, sanctions, strikes, escalation, and supply disruption before those effects are fully reflected in historical price patterns.

The objective of the notebook is therefore to explain one specific empirical pattern:

- why `Baseline` stays reasonably strong across both regimes
- why `Sent` improves much more during the war
- which features and feature families explain that difference

## Target and Evaluation Design

The target is **7-day excess return versus `XLE`**, using a **1.0% threshold** so the model is asked to predict meaningful relative moves rather than tiny noise.

The reporting windows are:

- **Pre-war**: April 14, 2025 to June 12, 2025
- **War**: June 13, 2025 to August 11, 2025

The pair is evaluated on the same broad overlap in both windows:

- **83** pre-war observations
- **89** war observations

The notebook is built from saved walk-forward out-of-sample prediction streams. So the plots are explaining an already-defined out-of-sample result, not an in-sample fit.

## Model Construction

There are two model objects in the analysis:

- **Baseline** = `lstm_base`
- **Sent** = `25% lstm_base + 75% lstm_sent`

`Baseline` is a broad market-and-event sequence model. Its main feature blocks are:

- returns, volatility, trend, and reversal
- oil, ETF, macro, and cross-asset context
- market structure such as beta, correlation, and residual relationships
- dated event and regime variables such as geopolitical-risk and statement-event weights

`lstm_sent` uses the same temporal backbone and adds a text block with features such as:

- geopolitical, shipping, sanctions, and supply-risk topic exposure
- supply-risk intensity and disruption proxies
- theory-driven `Agent` features such as political risk, company exposure, and war alignment

So `Sent` should be read as **Baseline plus text interpretation**, not as a completely separate architecture. That is important, because any war-period improvement can then be interpreted as incremental value from understanding the language rather than from changing the whole modeling setup.

In [55]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import yfinance as yf

warnings.filterwarnings("ignore")
px.defaults.template = "plotly_white"
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_colwidth", 120)


import importlib
import importlib.util
import subprocess

def ensure_parquet_engine() -> None:
    if importlib.util.find_spec("pyarrow") is not None or importlib.util.find_spec("fastparquet") is not None:
        return
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow", "-q"])
    importlib.invalidate_caches()

ensure_parquet_engine()


try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

# Load the saved study artifacts and prediction streams used throughout the analysis notebook.
def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'oil_projects').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the war_sentiment_study folder.')


PROJECT_DIR = find_project_dir(Path.cwd().resolve())
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'


prediction_files = {
    "lstm_base": ARTIFACT_DIR / "war_sentiment_lstm_pair_baseline_predictions.csv",
    "lstm_sent": ARTIFACT_DIR / "war_sentiment_lstm_pair_sentiment_predictions.csv",
}

# Merge the saved baseline and sent prediction streams onto the shared date-ticker overlap.
frames = []
for model_name, path in prediction_files.items():
    frame = pd.read_csv(path, parse_dates=["date"])[["date", "ticker", "target", "probability"]]
    frame = frame.rename(columns={"probability": model_name})
    frames.append(frame)

from functools import reduce
overlap = reduce(lambda left, right: left.merge(right, on=["date", "ticker", "target"]), frames)
overlap = overlap.sort_values(["date", "ticker"]).reset_index(drop=True)

PRE_START = pd.Timestamp("2025-04-14")
PRE_END = pd.Timestamp("2025-06-12")
WAR_START = pd.Timestamp("2025-06-13")
WAR_END = pd.Timestamp("2025-08-11")

period_map = np.where(
    (overlap["date"] >= PRE_START) & (overlap["date"] <= PRE_END),
    "Pre-war",
    np.where((overlap["date"] >= WAR_START) & (overlap["date"] <= WAR_END), "War", "Other"),
)
overlap["period"] = period_map
analysis = overlap[overlap["period"].isin(["Pre-war", "War"])].copy()

analysis["baseline_probability"] = analysis["lstm_base"]
analysis["sent_probability"] = 0.25 * analysis["lstm_base"] + 0.75 * analysis["lstm_sent"]
analysis["baseline_prediction"] = (analysis["baseline_probability"] >= 0.5).astype(int)
analysis["sent_prediction"] = (analysis["sent_probability"] >= 0.5).astype(int)
analysis["baseline_correct"] = (analysis["baseline_prediction"] == analysis["target"].astype(int)).astype(int)
analysis["sent_correct"] = (analysis["sent_prediction"] == analysis["target"].astype(int)).astype(int)
analysis["baseline_confidence"] = (analysis["baseline_probability"] - 0.5).abs() * 2
analysis["sent_confidence"] = (analysis["sent_probability"] - 0.5).abs() * 2

# Summarize the matched pair by regime so the first plot can compare pre-war and war behavior.
summary = pd.DataFrame(
    [
        {
            "model": "Baseline",
            "period": period,
            "accuracy": group["baseline_correct"].mean(),
            "n": len(group),
            "correct": int(group["baseline_correct"].sum()),
            "avg_confidence": group["baseline_confidence"].mean(),
        }
        for period, group in analysis.groupby("period")
    ]
    + [
        {
            "model": "Sent",
            "period": period,
            "accuracy": group["sent_correct"].mean(),
            "n": len(group),
            "correct": int(group["sent_correct"].sum()),
            "avg_confidence": group["sent_confidence"].mean(),
        }
        for period, group in analysis.groupby("period")
    ]
)

selected_metrics = {
    "project_name": "war_sentiment_lstm_pair_selected",
    "model_definition": "Baseline is raw baseline_lstm. Sent is 25% baseline_lstm probability plus 75% sentiment_lstm probability.",
    "baseline": {
        period: {
            "accuracy": float(group["baseline_correct"].mean()),
            "predictions": int(len(group)),
            "total_possible": int(len(group)),
        }
        for period, group in analysis.groupby("period")
    },
    "sent": {
        period: {
            "accuracy": float(group["sent_correct"].mean()),
            "predictions": int(len(group)),
            "total_possible": int(len(group)),
        }
        for period, group in analysis.groupby("period")
    },
}
(ARTIFACT_DIR / "war_sentiment_lstm_pair_selected_metrics.json").write_text(json.dumps(selected_metrics, indent=2))


671

## Selected Broad Pair: Accuracy Before the War and During the War

This is the starting point for the whole analysis. Both models are evaluated on the same date-ticker overlap, so the bars are not comparing different samples. The useful thing to watch is whether the distance between `Baseline` and `Sent` changes after the regime break. If text really becomes more valuable in a war-driven market, the gap should widen during the war window.

In [56]:
# Start with the matched headline comparison: the same overlap, split into pre-war and war.
fig = px.bar(
    summary,
    x="model",
    y="accuracy",
    color="period",
    barmode="group",
    text=summary["accuracy"].round(3),
    hover_data=["n", "correct", "avg_confidence"],
    title="Selected Broad Pair: Accuracy Before the War and During the War",
    color_discrete_map={"Pre-war": "#2f855a", "War": "#c53030"},
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="Accuracy", xaxis_title="")
fig.show()


Both models are already strong before the war, but only `Sent` improves materially during the war. `Baseline` moves from **0.6145** to **0.6067**, while `Sent` moves from **0.6386** to **0.7079**. The `Sent - Baseline` edge therefore widens from about **+2.4 percentage points** pre-war to about **+10.1 percentage points** during war.

This is the core empirical fact the rest of the notebook needs to explain.

## Where the Models Agree and Disagree

After the headline accuracy gap, the next question is whether `Sent` is actually finding different correct cases or simply looking more confident on the same calls. The most important bars are the one-model-only wins: they show where one model is right and the other is wrong. Those cases are the cleanest evidence of incremental signal.

In [57]:
# Label each observation by whether Baseline and Sent are both right, both wrong, or disagree.
behavior = analysis.copy()
behavior['agreement'] = np.select(
    [
        (behavior['baseline_correct'] == 1) & (behavior['sent_correct'] == 1),
        (behavior['baseline_correct'] == 1) & (behavior['sent_correct'] == 0),
        (behavior['baseline_correct'] == 0) & (behavior['sent_correct'] == 1),
    ],
    ['Both correct', 'Baseline only', 'Sent only'],
    default='Both wrong',
)

prob_long = behavior.melt(
    id_vars=['period', 'ticker', 'date'],
    value_vars=['baseline_probability', 'sent_probability'],
    var_name='model',
    value_name='probability',
)
prob_long['model'] = prob_long['model'].map({
    'baseline_probability': 'Baseline',
    'sent_probability': 'Sent',
})



In [58]:
agreement_counts = (
    behavior.groupby(['period', 'agreement'])
    .size()
    .reset_index(name='count')
)

fig = px.bar(
    agreement_counts,
    x='period',
    y='count',
    color='agreement',
    barmode='stack',
    text='count',
    title='Where the Models Agree and Disagree',
    color_discrete_map={
        'Both correct': '#2f855a',
        'Baseline only': '#355C7D',
        'Sent only': '#C06C84',
        'Both wrong': '#a0aec0',
    },
)
fig.update_traces(textposition='inside')
fig.update_layout(xaxis_title='', yaxis_title='Prediction count')
fig.show()


The war outperformance is real incremental information. During the war, `Sent` wins **14** cases that `Baseline` misses, while `Baseline` wins only **5** cases that `Sent` misses. Both are correct on **49** war cases. So the edge is coming from extra `Sent` wins, not from duplicated baseline calls.

## Official Event Days Versus Non-Event Days

Once the `Sent` edge is visible, we need to check where it appears. The thesis says the text layer should help most when dated geopolitical supply information enters the market. This plot separates official event days from other war days, so it is best read as mechanism evidence rather than the main headline metric.

In [59]:
# Split war-period observations into event days and non-event days using the dated statement-event file.
event_sources = pd.read_csv(ARTIFACT_DIR / 'statement_event_sources.csv', parse_dates=['date'])
event_sources = event_sources[(event_sources['date'] >= WAR_START) & (event_sources['date'] <= WAR_END)].copy()
event_days = event_sources[['date']].drop_duplicates().assign(event_day=1)

war_event_analysis = behavior[behavior['period'] == 'War'].merge(event_days, on='date', how='left')
war_event_analysis['event_day'] = war_event_analysis['event_day'].fillna(0).astype(int)
war_event_analysis['subset'] = np.where(war_event_analysis['event_day'] == 1, 'Official event day', 'No official event')

event_split = pd.concat(
    [
        war_event_analysis.groupby('subset', as_index=False).agg(
            accuracy=('baseline_correct', 'mean'),
            n=('ticker', 'size'),
        ).assign(model='Baseline'),
        war_event_analysis.groupby('subset', as_index=False).agg(
            accuracy=('sent_correct', 'mean'),
            n=('ticker', 'size'),
        ).assign(model='Sent'),
    ],
    ignore_index=True,
)
event_split['label'] = event_split.apply(lambda row: f"{row['accuracy']:.3f}<br>n={int(row['n'])}", axis=1)

fig = px.bar(
    event_split,
    x='subset',
    y='accuracy',
    color='model',
    barmode='group',
    text='label',
    title='War Accuracy on Official Event Days Versus Non-Event Days',
    color_discrete_map={'Baseline': '#2f855a', 'Sent': '#c53030'},
)
fig.update_traces(textposition='outside')
fig.update_layout(height=560, xaxis_title='', yaxis_title='Accuracy')
fig.show()


The gap is much larger on official event days. In the current run, `Sent` reaches **0.750** on event days compared with **0.250** for `Baseline`, while on non-event war days the gap is much smaller at roughly **0.704** compared with **0.642**. That is exactly the kind of concentration the research objective predicts.

## War-Period Sent Edge Around Official Events

The event-day split shows that the edge is concentrated, but it does not show timing. Here the daily `Sent - Baseline` bars are placed next to official event markers. Positive bars near those markers suggest that the model difference is connected to the arrival of war-related information, not just to a general war-period average.

In [60]:
# Align daily Sent-minus-Baseline edge with the official event timeline and nearby WTI moves.
event_sources = pd.read_csv(ARTIFACT_DIR / 'statement_event_sources.csv', parse_dates=['date'])
event_sources = event_sources[(event_sources['date'] >= WAR_START) & (event_sources['date'] <= WAR_END)].copy()
event_daily = event_sources.groupby('date', as_index=False).agg(
    total_weight=('weight', 'sum'),
    title=('title', 'first'),
)

war_daily_edge = (
    behavior[behavior['period'] == 'War']
    .groupby('date', as_index=False)
    .agg(
        baseline_accuracy=('baseline_correct', 'mean'),
        sent_accuracy=('sent_correct', 'mean'),
    )
)
war_daily_edge['sent_minus_baseline'] = war_daily_edge['sent_accuracy'] - war_daily_edge['baseline_accuracy']

crude = yf.download('CL=F', start=str(WAR_START.date()), end=str((WAR_END + pd.Timedelta(days=2)).date()), progress=False, auto_adjust=False)
if isinstance(crude.columns, pd.MultiIndex):
    crude.columns = crude.columns.get_level_values(0)
crude.columns = [str(column).title() for column in crude.columns]
crude.index = pd.to_datetime(crude.index).tz_localize(None)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.45, 0.55],
    subplot_titles=('WTI crude', 'Daily sent edge with official event markers'),
)

fig.add_trace(
    go.Scatter(
        x=crude.index,
        y=crude['Close'],
        mode='lines',
        name='WTI close',
        line=dict(color='#4a5568', width=2),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=war_daily_edge['date'],
        y=war_daily_edge['sent_minus_baseline'],
        name='Sent - Baseline',
        marker_color=['#c53030' if value >= 0 else '#2f855a' for value in war_daily_edge['sent_minus_baseline']],
        hovertemplate='Date=%{x}<br>Sent minus baseline=%{y:.3f}<extra></extra>',
    ),
    row=2,
    col=1,
)

if not event_daily.empty:
    y_marker = max(0.05, war_daily_edge['sent_minus_baseline'].max() + 0.03)
    fig.add_trace(
        go.Scatter(
            x=event_daily['date'],
            y=[y_marker] * len(event_daily),
            mode='markers',
            marker=dict(color='#dd6b20', size=11, symbol='diamond'),
            name='Official event',
            text=event_daily['title'],
            hovertemplate='Date=%{x}<br>%{text}<extra></extra>',
        ),
        row=2,
        col=1,
    )

fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)
fig.update_layout(
    title='War-Period Sent Edge Around Official Events',
    height=760,
    legend=dict(orientation='h', y=1.08, x=0.01),
)
fig.update_yaxes(title_text='WTI close', row=1, col=1)
fig.update_yaxes(title_text='Sent minus baseline', row=2, col=1)
fig.show()


The `Sent` edge is not only larger during war; it is concentrated around the kind of dated information flow the study is about. This connects back to the event-day result where `Sent` reached **0.750** accuracy on official event days while `Baseline` reached **0.250**. That timing supports the mechanism before we even inspect individual features.

## Predicted Probability Distribution by Model and Period

This plot is a behavior check after the event evidence. Probabilities close to `0.5` are low-conviction calls, while probabilities farther away from `0.5` are stronger directional views. A wider or shifted distribution is not automatically better; it only matters if it produces extra correct calls, which the previous plots already tested.

In [61]:
# Compare the probability distributions to see how model conviction changes across regimes.
fig = px.violin(
    prob_long,
    x='model',
    y='probability',
    color='period',
    box=True,
    points=False,
    title='Predicted Probability Distribution by Model and Period',
    color_discrete_map={'Pre-war': '#2f855a', 'War': '#c53030'},
)
fig.update_layout(xaxis_title='', yaxis_title='Predicted probability', height=620)
fig.show()


The war-period `Sent` edge is not just a confidence artifact. The distribution changes matter because they turn into extra correct calls: during the war, `Sent` wins **14** cases that `Baseline` misses, while `Baseline` wins only **5** cases that `Sent` misses.

## Feature Sets and What They Are Meant to Capture

The next sections move from model-level behavior to feature-level explanation. The goal is to separate three questions:

1. Which individual features carry standalone information?
2. Which features the trained models actually weight?
3. Whether the war improvement comes from different feature families becoming informative.

The next plots answer two different questions. The feature-signal plots ask whether a feature has any standalone directional information on its own. The weighting plots ask what the trained network actually leans on once all features compete inside the model.

The baseline panel is dominated by structured market and event features: returns, volatility, correlation structure, oil and ETF context, and official event flags. The sent side adds semantic text features such as geopolitical share, shipping share, supply-risk intensity, and theory-driven agent features that try to encode political risk, company exposure, and war alignment.

### What `Sentiment` and `Agent` Mean

These two families are related, but they are not the same thing.

`Sentiment` features are direct text-interpretation variables. They describe what the language is about and how strongly it points toward supply risk. In this project that includes features like `geopolitics_share`, `shipping_share`, `sanctions_share`, `supply_risk_intensity`, and `supply_disruption_share`.

`Agent` features are theory-driven composites built on top of text, event context, and market context. They try to translate raw language into economically meaningful states such as political risk, company exposure, war alignment, disclosure relevance, and uncertainty confirmation. In this project that includes features like `agent_political_risk`, `agent_company_exposure`, `agent_market_stress`, `agent_disclosure_signal`, `agent_war_alignment`, and `agent_uncertainty_confirmation`.

The practical difference is:

- `Sentiment` asks what the text says.
- `Agent` asks what that text means for an oil-sensitive prediction problem.

That distinction matters in the weighting plots. When the sent model shifts weight into `Sentiment`, it is leaning more on war-related language itself. When it shifts weight into `Agent`, it is leaning more on interpreted geopolitical and exposure signals rather than on raw wording alone.

In [62]:
# Load the saved daily panels and prepare the single-feature diagnostic frames used in the transparency plots.
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

from sklearn.metrics import roc_auc_score

quant_panel = pd.read_parquet(PROJECT_DIR / 'data/processed/quant_panel.parquet')
news_panel = pd.read_parquet(PROJECT_DIR / 'data/processed/news_panel.parquet')

quant_panel['date'] = pd.to_datetime(quant_panel['date'])
news_panel['date'] = pd.to_datetime(news_panel['date'])

quant_panel['period'] = np.where(
    (quant_panel['date'] >= PRE_START) & (quant_panel['date'] <= PRE_END),
    'Pre-war',
    np.where((quant_panel['date'] >= WAR_START) & (quant_panel['date'] <= WAR_END), 'War', 'Other'),
)
news_panel['period'] = np.where(
    (news_panel['date'] >= PRE_START) & (news_panel['date'] <= PRE_END),
    'Pre-war',
    np.where((news_panel['date'] >= WAR_START) & (news_panel['date'] <= WAR_END), 'War', 'Other'),
)

baseline_features = [
    'ret_1',
    'ret_5',
    'vol_10',
    'beta_xle_20',
    'corr_xle_20',
    'cl_ret',
    'xle_ret',
    'official_event_weight',
    'market_stress_day',
]

sent_features = [
    'geopolitics_share',
    'shipping_share',
    'supply_risk_intensity',
    'agent_war_alignment',
    'agent_market_stress',
    'ovx_supply_stress',
    'gpr_threat',
    'gpr_threat_share',
]

def feature_signal_table(frame: pd.DataFrame, features: list[str], label: str) -> pd.DataFrame:
    rows = []
    for feature in features:
        if feature not in frame.columns:
            rows.append(
                {
                    'model': label,
                    'feature': feature,
                    'period': 'Missing',
                    'auc': np.nan,
                    'signal': np.nan,
                    'n': 0,
                    'unique_values': 0,
                    'non_zero_share': np.nan,
                    'status': 'missing',
                    'reason': 'column missing from saved panel',
                }
            )
            continue
        for period in ['Pre-war', 'War']:
            sub = frame[frame['period'] == period][[feature, 'target']].replace([np.inf, -np.inf], np.nan).dropna()
            n = len(sub)
            unique_values = int(sub[feature].nunique()) if n else 0
            non_zero_share = float((sub[feature] != 0).mean()) if n else np.nan
            if n < 20:
                status = 'inactive'
                reason = 'too few rows'
                auc = np.nan
            elif sub['target'].nunique() < 2:
                status = 'inactive'
                reason = 'target has one class'
                auc = np.nan
            elif unique_values < 2:
                status = 'inactive'
                reason = 'feature is constant in this period'
                auc = np.nan
            else:
                status = 'active'
                reason = 'ok'
                auc = float(roc_auc_score(sub['target'].astype(int), sub[feature]))
            rows.append(
                {
                    'model': label,
                    'feature': feature,
                    'period': period,
                    'auc': auc,
                    'signal': auc - 0.5 if pd.notna(auc) else np.nan,
                    'n': n,
                    'unique_values': unique_values,
                    'non_zero_share': non_zero_share,
                    'status': status,
                    'reason': reason,
                }
            )
    return pd.DataFrame(rows)

baseline_signal = feature_signal_table(quant_panel, baseline_features, 'Baseline panel')
sent_signal = feature_signal_table(news_panel, sent_features, 'Sent panel')
feature_signal = pd.concat([baseline_signal, sent_signal], ignore_index=True)
plotworthy_feature_summary = (
    feature_signal[feature_signal['period'].isin(['Pre-war', 'War'])]
    .pivot_table(
        index=['model', 'feature'],
        columns='period',
        values=['auc', 'signal', 'non_zero_share'],
        aggfunc='first',
    )
    .reset_index()
)
plotworthy_feature_summary.columns = [
    'model',
    'feature',
    'auc_pre',
    'auc_war',
    'non_zero_pre',
    'non_zero_war',
    'signal_pre',
    'signal_war',
]
plotworthy_feature_summary['war_minus_pre'] = plotworthy_feature_summary['signal_war'] - plotworthy_feature_summary['signal_pre']
plotworthy_feature_summary['max_abs_signal'] = plotworthy_feature_summary[['signal_pre', 'signal_war']].abs().max(axis=1)
plotworthy_feature_summary['info_pre'] = plotworthy_feature_summary['signal_pre'].abs()
plotworthy_feature_summary['info_war'] = plotworthy_feature_summary['signal_war'].abs()
plotworthy_feature_summary['min_non_zero_share'] = plotworthy_feature_summary[['non_zero_pre', 'non_zero_war']].min(axis=1)
plotworthy_feature_summary = plotworthy_feature_summary[
    plotworthy_feature_summary[['auc_pre', 'auc_war', 'non_zero_pre', 'non_zero_war']].notna().all(axis=1)
].copy()
plotworthy_feature_summary = plotworthy_feature_summary[
    (plotworthy_feature_summary['min_non_zero_share'] > 0.01)
    & (plotworthy_feature_summary['max_abs_signal'] >= 0.01)
].copy()

plotworthy_feature_signal = feature_signal.merge(
    plotworthy_feature_summary[['model', 'feature']],
    on=['model', 'feature'],
    how='inner',
)



In [63]:
# Define the helper routines that score one feature at a time and format the transparency plots.
def plot_feature_heatmap(data: pd.DataFrame, title: str, color_scale: str = 'RdBu'):
    plot_data = data.copy()
    pivot_auc = plot_data.pivot(index='feature', columns='period', values='auc').reindex(sorted(plot_data['feature'].unique()))
    pivot_n = plot_data.pivot(index='feature', columns='period', values='n').reindex(pivot_auc.index)
    pivot_signal = plot_data.pivot(index='feature', columns='period', values='signal').reindex(pivot_auc.index)
    text = pivot_auc.copy().astype(object)
    for feature in pivot_auc.index:
        for period in pivot_auc.columns:
            auc = pivot_auc.loc[feature, period]
            n = pivot_n.loc[feature, period]
            signal = pivot_signal.loc[feature, period]
            text.loc[feature, period] = f'{auc:.3f}<br>edge={signal:+.3f}<br>n={int(n)}'

    fig = go.Figure(
        data=go.Heatmap(
            z=pivot_auc.values - 0.5,
            x=pivot_auc.columns,
            y=pivot_auc.index,
            text=text.values,
            texttemplate='%{text}',
            textfont={'size': 11},
            colorscale=color_scale,
            zmin=-0.15,
            zmax=0.15,
            colorbar=dict(title='AUC - 0.50'),
            hovertemplate='Feature=%{y}<br>Period=%{x}<br>AUC=%{text}<extra></extra>',
        )
    )
    fig.update_layout(title=title, xaxis_title='', yaxis_title='Feature', height=520)
    fig.show()



### Baseline Standalone Signal

Before looking at what the network weights, it helps to see what the baseline features can do alone. This is a signed standalone-edge view. A negative value does not mean the feature is useless; it means the feature points in the opposite direction and can still help once the model learns that sign.

In [64]:
# Show which baseline features carry standalone directional information before and during war.
plot_feature_heatmap(
    plotworthy_feature_signal[plotworthy_feature_signal['model'] == 'Baseline panel'],
    'Baseline Feature Signal: Pre-war vs War',
    color_scale='Greens',
)

There is no single baseline feature that explains the full model. The baseline is being built from many weak market features, and several of the strongest war features such as `cl_ret`, `vol_10`, `beta_xle_20`, and `xle_ret` carry information mainly with an inverse sign. That is why the baseline can stay reasonably accurate even when the heatmap does not look visually positive.

### Sent Standalone Signal

This is the cleanest feature-level check of the text thesis. If war-related language matters more during the shock, some text-linked features should become more informative in the war window. The sign still matters, but the main thing to watch is whether war-salient text features improve relative to pre-war.

In [65]:
# Show which sent-side text features gain or lose standalone information across regimes.
plot_feature_heatmap(
    plotworthy_feature_signal[plotworthy_feature_signal['model'] == 'Sent panel'],
    'Sent Feature Signal: Pre-war vs War',
    color_scale='Reds',
)

The improvement is concentrated in the war-salience text features. `geopolitics_share`, `shipping_share`, and `supply_risk_intensity` move from slightly negative or neutral pre-war to slightly positive in war. That is small in isolation, but directionally important because those are exactly the features the war thesis would predict.

### How Feature Signal Moves from Pre-war to War

The heatmaps separate the models, while this view puts the regime shift itself at the center. Green marks pre-war and red marks war. When a feature moves to the right, it becomes more useful in the war window; when it moves left, it loses standalone usefulness.

In [66]:
# Compare the active features across the pre-war and war periods.
regime_map = plotworthy_feature_summary.copy()
fig = make_subplots(
    rows=1,
    cols=2,
    shared_xaxes=True,
    subplot_titles=('Baseline', 'Sent'),
    horizontal_spacing=0.12,
)

for col, model_key in enumerate(['Baseline panel', 'Sent panel'], start=1):
    sub = regime_map[regime_map['model'] == model_key].copy()
    sub = sub.sort_values(['war_minus_pre', 'signal_war', 'feature'], ascending=[True, False, True]).reset_index(drop=True)
    sub['feature_label'] = sub['feature']
    for _, row in sub.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[row['signal_pre'], row['signal_war']],
                y=[row['feature_label'], row['feature_label']],
                mode='lines',
                line=dict(color='rgba(160,174,192,0.7)', width=2),
                hoverinfo='skip',
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    fig.add_trace(
        go.Scatter(
            x=sub['signal_pre'],
            y=sub['feature_label'],
            mode='markers',
            name='Pre-war',
            marker=dict(color='#2f855a', size=11, symbol='circle'),
            legendgroup='period',
            showlegend=(col == 1),
            hovertemplate='Feature=%{y}<br>Pre-war edge=%{x:.3f}<extra></extra>',
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=sub['signal_war'],
            y=sub['feature_label'],
            mode='markers',
            name='War',
            marker=dict(color='#c53030', size=11, symbol='diamond'),
            legendgroup='period',
            showlegend=(col == 1),
            hovertemplate='Feature=%{y}<br>War edge=%{x:.3f}<extra></extra>',
        ),
        row=1,
        col=col,
    )

fig.update_xaxes(title_text='Single-feature edge', zeroline=True, zerolinecolor='gray', row=1, col=1)
fig.update_xaxes(title_text='Single-feature edge', zeroline=True, zerolinecolor='gray', row=1, col=2)
fig.update_layout(
    title='How Standalone Feature Signal Moves from Pre-war to War',
    height=720,
    margin=dict(t=90, l=60, r=40, b=40),
    legend=dict(orientation='h', y=1.08, x=0.01),
)
fig.show()


The baseline features mostly stay at or below zero in war, while several war-salience `Sent` features move to the right. That gives the first compact feature-level explanation for why `Sent` improves and `Baseline` does not.

## Feature Weighting

The next plots move from standalone signal to model usage. A feature can have weak standalone AUC and still receive meaningful weight if it works well in combination with other inputs.

The weighting view is extracted from a pre-war reference fit. For the backbone, weight share comes from the first-layer LSTM input weights adjusted for actual feature variation. For the sent-only text block, weight share comes from the static head and its connection into the main prediction path.

In [67]:
# Load the saved pre-war reference weights used to inspect how the trained models allocate attention.
weight_path = ARTIFACT_DIR / 'war_sentiment_lstm_pair_reference_weights.json'
if not weight_path.exists():
    raise FileNotFoundError('Run 01_training_lstm_pair.ipynb first to generate the reference weight artifact.')

weight_payload = json.loads(weight_path.read_text())
baseline_weight_df = pd.DataFrame(weight_payload['baseline_sequence_weights'])
sent_sequence_weight_df = pd.DataFrame(weight_payload['sent_sequence_weights'])
sent_static_weight_df = pd.DataFrame(weight_payload['sent_static_weights'])

def prepare_weight_table(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    out = frame.copy()
    out = out[out['weight_share'] > 0].copy()
    out = out.sort_values('weight_share', ascending=False).reset_index(drop=True)
    out['rank'] = np.arange(1, len(out) + 1)
    out['cumulative_share'] = out['weight_share'].cumsum()
    out['model'] = label
    return out

baseline_ranked = prepare_weight_table(baseline_weight_df, 'Baseline backbone')
sent_sequence_ranked = prepare_weight_table(sent_sequence_weight_df, 'Sent backbone')
sent_static_ranked = prepare_weight_table(sent_static_weight_df, 'Sent text block')

def feature_family(feature: str) -> str:
    if feature.startswith('ret_') or feature in {'prev_ret_1', 'prev_ret_5', 'residual_oil_1', 'residual_xle_1'}:
        return 'Returns'
    if feature.startswith('vol_') or feature in {'range_1', 'volume_z_20'}:
        return 'Volatility'
    if feature.startswith('beta_') or feature.startswith('corr_') or feature.startswith('shock_'):
        return 'Structure'
    if feature in {'CL=F_ret_1', 'CL=F_ret_5', 'BZ=F_ret_1', 'BZ=F_ret_5', 'cl_ret', 'bz_ret', 'xle_ret', 'vix_ret', 'SPY_ret_1', 'SPY_ret_5', '^VIX_ret_1', '^VIX_ret_5', 'UUP_ret_1', 'UUP_ret_5'}:
        return 'Macro'
    if feature.startswith('filing_') or feature in {'avg_text_length', 'novelty_mean'}:
        return 'Disclosure'
    if feature.startswith('gpr_') or feature.startswith('official_') or feature.startswith('statement_') or feature == 'market_stress_day':
        return 'Policy/Event'
    if feature.startswith('agent_'):
        return 'Agent'
    if 'sentiment' in feature or feature.endswith('_share') or 'supply' in feature or 'shipping' in feature or 'sanction' in feature:
        return 'Sentiment'
    return 'Other'

sent_total_weights = pd.concat(
    [
        sent_sequence_weight_df[['feature', 'effective_weight']],
        sent_static_weight_df[['feature', 'effective_weight']],
    ],
    ignore_index=True,
)
sent_total_weights = sent_total_weights[sent_total_weights['effective_weight'] > 0].copy()
sent_total_weights['family'] = sent_total_weights['feature'].map(feature_family)
sent_total_family = sent_total_weights.groupby('family', as_index=False)['effective_weight'].sum()
sent_total_family['Sent total'] = sent_total_family['effective_weight'] / sent_total_family['effective_weight'].sum()
sent_total_family = sent_total_family.drop(columns=['effective_weight'])

baseline_family = baseline_ranked.copy()
baseline_family['family'] = baseline_family['feature'].map(feature_family)
baseline_family = baseline_family.groupby('family', as_index=False)['weight_share'].sum().rename(columns={'weight_share': 'Baseline'})

family_compare = baseline_family.merge(sent_total_family, on='family', how='outer').fillna(0.0)
family_compare = family_compare[(family_compare['Baseline'] > 0) | (family_compare['Sent total'] > 0)].copy()
family_compare['sent_minus_baseline'] = family_compare['Sent total'] - family_compare['Baseline']


### Top Weighted Features Inside Each Model Block

Standalone signal is only half the story. This plot shows what the trained model actually emphasizes. Each panel is normalized within its own model block, so the comparison is about which inputs each block leans on most relative to its own alternatives.

In [68]:
# Keep only the top-weighted features from each model block so the comparison stays readable.
top_frames = []
for frame, block in [
    (baseline_ranked.head(10), 'Baseline backbone'),
    (sent_sequence_ranked.head(10), 'Sent backbone'),
    (sent_static_ranked.head(10), 'Sent text block'),
]:
    tmp = frame.copy().sort_values('weight_share', ascending=True)
    tmp['block'] = block
    top_frames.append(tmp)
top_weight_df = pd.concat(top_frames, ignore_index=True)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=('Baseline backbone', 'Sent backbone', 'Sent text block'),
)

block_meta = [
    ('Baseline backbone', '#2f855a', 1),
    ('Sent backbone', '#c53030', 2),
    ('Sent text block', '#8b1e3f', 3),
]
for block, color, row in block_meta:
    sub = top_weight_df[top_weight_df['block'] == block]
    fig.add_trace(
        go.Bar(
            x=sub['weight_share'],
            y=sub['feature'],
            orientation='h',
            name=block,
            marker_color=color,
            hovertemplate='Feature=%{y}<br>Weight share=%{x:.3%}<extra></extra>',
            showlegend=False,
        ),
        row=row,
        col=1,
    )

fig.update_layout(
    title='Top Weighted Features Inside Each Model Block',
    height=1150,
    margin=dict(t=90, l=60, r=40, b=40),
)
fig.update_xaxes(title_text='Relative weight share', tickformat='.1%', row=1, col=1)
fig.update_xaxes(title_text='Relative weight share', tickformat='.1%', row=2, col=1)
fig.update_xaxes(title_text='Relative weight share', tickformat='.1%', row=3, col=1)
fig.show()


The baseline backbone is led by structured event and market features. Its largest reference weights include `statement_shipping_weight_3d` at about **2.9%** and `statement_event_weight` at about **2.6%**. The `Sent` text block puts more concentrated capacity into war-meaning variables, including `supply_disruption_share` at about **7.7%**, `agent_company_exposure` at about **7.3%**, and `geopolitics_share` at about **7.2%**. So the model architecture itself is aligned with the research idea: event occurrence sits in the backbone, and interpretation of meaning sits in the text block.

### Weighted Features Versus Standalone Signal

This is where model usage and feature information meet. The view uses absolute information magnitude rather than signed edge, so a larger value means the feature carries information even if the model uses it with a negative sign. That is important because many useful market features work through inverse relationships.

In [69]:
# Join weight share with standalone information so we can see whether emphasized features actually carry signal.
baseline_signal_plot = (
    plotworthy_feature_summary[plotworthy_feature_summary['model'] == 'Baseline panel']
    .merge(baseline_ranked[['feature', 'weight_share']], on='feature', how='inner')
    .sort_values('weight_share', ascending=False)
    .head(6)
    .assign(model_label='Baseline')
)

sent_total_feature_weights = (
    sent_total_weights.groupby('feature', as_index=False)['effective_weight'].sum()
)
sent_total_feature_weights['weight_share'] = sent_total_feature_weights['effective_weight'] / sent_total_feature_weights['effective_weight'].sum()
sent_signal_plot = (
    plotworthy_feature_summary[plotworthy_feature_summary['model'] == 'Sent panel']
    .merge(sent_total_feature_weights[['feature', 'weight_share']], on='feature', how='inner')
    .sort_values('weight_share', ascending=False)
    .head(6)
    .assign(model_label='Sent')
)

weight_signal_plot = pd.concat([baseline_signal_plot, sent_signal_plot], ignore_index=True).copy()
weight_signal_plot['max_info'] = weight_signal_plot[['info_pre', 'info_war']].max(axis=1)

fig = make_subplots(
    rows=1,
    cols=2,
    shared_xaxes=True,
    subplot_titles=('Baseline', 'Sent'),
    horizontal_spacing=0.12,
)

for col, model_name in enumerate(['Baseline', 'Sent'], start=1):
    sub = (
        weight_signal_plot[weight_signal_plot['model_label'] == model_name]
        .sort_values(['max_info', 'weight_share', 'feature'], ascending=[True, True, True])
        .reset_index(drop=True)
    )
    for _, row in sub.iterrows():
        fig.add_trace(
            go.Scatter(
                x=[row['info_pre'], row['info_war']],
                y=[row['feature'], row['feature']],
                mode='lines',
                line=dict(color='rgba(160,174,192,0.75)', width=2),
                hoverinfo='skip',
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    fig.add_trace(
        go.Scatter(
            x=sub['info_pre'],
            y=sub['feature'],
            mode='markers+text',
            name='Pre-war',
            marker=dict(color='#2f855a', size=11, symbol='circle'),
            text=[f"{value:.3f}" for value in sub['info_pre']],
            textposition='middle left',
            textfont=dict(size=10, color='#2f855a'),
            customdata=np.stack([sub['weight_share'], sub['signal_pre'], sub['signal_war']], axis=1),
            hovertemplate='Feature=%{y}<br>Pre-war information=%{x:.3f}<br>Weight share=%{customdata[0]:.3%}<br>Pre-war signed edge=%{customdata[1]:.3f}<br>War signed edge=%{customdata[2]:.3f}<extra></extra>',
            legendgroup='period',
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=sub['info_war'],
            y=sub['feature'],
            mode='markers+text',
            name='War',
            marker=dict(color='#c53030', size=11, symbol='diamond'),
            text=[f"{value:.3f}" for value in sub['info_war']],
            textposition='middle right',
            textfont=dict(size=10, color='#c53030'),
            customdata=np.stack([sub['weight_share'], sub['signal_pre'], sub['signal_war']], axis=1),
            hovertemplate='Feature=%{y}<br>War information=%{x:.3f}<br>Weight share=%{customdata[0]:.3%}<br>Pre-war signed edge=%{customdata[1]:.3f}<br>War signed edge=%{customdata[2]:.3f}<extra></extra>',
            legendgroup='period',
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )

fig.update_xaxes(title_text='Standalone information magnitude', row=1, col=1)
fig.update_xaxes(title_text='Standalone information magnitude', row=1, col=2)
fig.update_layout(
    title='Do the Most Heavily Weighted Features Carry Information?',
    height=540,
    margin=dict(t=90, l=40, r=40, b=40),
    legend=dict(orientation='h', y=1.08, x=0.01),
)
fig.show()


For the baseline, heavily weighted features such as `vol_10`, `cl_ret`, and `xle_ret` do carry information, but much of that information is inverse or unstable in war. For `Sent`, some of the strongest weighted inputs still come from the shared event backbone, but the war-linked text variables become meaningfully informative in the war regime. That is a direct explanation for why the war edge widens without requiring any single feature to dominate on its own.

### Weighted Information by Feature Family

Individual features are noisy, so this plot rolls them into families. The value is weighted information magnitude, not signed signal. Larger bars mean that a family is both emphasized by the model and informative enough to matter.

In [70]:
# Roll feature-level weight and information into families to show where usable signal is coming from.
baseline_family_signal = (
    plotworthy_feature_summary[plotworthy_feature_summary['model'] == 'Baseline panel']
    .merge(baseline_ranked[['feature', 'weight_share']], on='feature', how='inner')
    .assign(family=lambda df: df['feature'].map(feature_family))
)
baseline_family_signal = pd.concat(
    [
        baseline_family_signal.groupby('family', as_index=False).apply(
            lambda g: pd.Series({
                'weighted_information': (g['weight_share'] * g['info_pre']).sum(),
                'weighted_signed_edge': (g['weight_share'] * g['signal_pre']).sum(),
                'period': 'Pre-war',
                'model': 'Baseline',
            })
        ),
        baseline_family_signal.groupby('family', as_index=False).apply(
            lambda g: pd.Series({
                'weighted_information': (g['weight_share'] * g['info_war']).sum(),
                'weighted_signed_edge': (g['weight_share'] * g['signal_war']).sum(),
                'period': 'War',
                'model': 'Baseline',
            })
        ),
    ],
    ignore_index=True,
)

sent_family_signal = (
    plotworthy_feature_summary[plotworthy_feature_summary['model'] == 'Sent panel']
    .merge(sent_total_feature_weights[['feature', 'weight_share']], on='feature', how='inner')
    .assign(family=lambda df: df['feature'].map(feature_family))
)
sent_family_signal = pd.concat(
    [
        sent_family_signal.groupby('family', as_index=False).apply(
            lambda g: pd.Series({
                'weighted_information': (g['weight_share'] * g['info_pre']).sum(),
                'weighted_signed_edge': (g['weight_share'] * g['signal_pre']).sum(),
                'period': 'Pre-war',
                'model': 'Sent',
            })
        ),
        sent_family_signal.groupby('family', as_index=False).apply(
            lambda g: pd.Series({
                'weighted_information': (g['weight_share'] * g['info_war']).sum(),
                'weighted_signed_edge': (g['weight_share'] * g['signal_war']).sum(),
                'period': 'War',
                'model': 'Sent',
            })
        ),
    ],
    ignore_index=True,
)

family_signal_plot = pd.concat([baseline_family_signal, sent_family_signal], ignore_index=True)
family_signal_plot = family_signal_plot[family_signal_plot['weighted_information'] > 0].copy()
family_order = (
    family_signal_plot.groupby('family', as_index=False)['weighted_information']
    .max()
    .sort_values('weighted_information', ascending=True)['family']
    .tolist()
)

fig = make_subplots(
    rows=1,
    cols=2,
    shared_xaxes=True,
    subplot_titles=('Baseline', 'Sent'),
    horizontal_spacing=0.12,
)

for col, model_name in enumerate(['Baseline', 'Sent'], start=1):
    sub = family_signal_plot[family_signal_plot['model'] == model_name].copy()
    pivot = (
        sub.pivot(index='family', columns='period', values=['weighted_information', 'weighted_signed_edge'])
        .reindex(family_order)
        .dropna(subset=[('weighted_information', 'Pre-war'), ('weighted_information', 'War')], how='all')
    )
    labels = pivot.index.tolist()
    pre_info = pivot[('weighted_information', 'Pre-war')].fillna(0).to_numpy()
    war_info = pivot[('weighted_information', 'War')].fillna(0).to_numpy()
    pre_edge = pivot[('weighted_signed_edge', 'Pre-war')].fillna(0).to_numpy()
    war_edge = pivot[('weighted_signed_edge', 'War')].fillna(0).to_numpy()

    for family, x0, x1 in zip(labels, pre_info, war_info):
        fig.add_trace(
            go.Scatter(
                x=[x0, x1],
                y=[family, family],
                mode='lines',
                line=dict(color='rgba(160,174,192,0.75)', width=2),
                hoverinfo='skip',
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    fig.add_trace(
        go.Scatter(
            x=pre_info,
            y=labels,
            mode='markers+text',
            name='Pre-war',
            marker=dict(color='#2f855a', size=12, symbol='circle'),
            text=[f"{value:.4f}" for value in pre_info],
            textposition='middle left',
            textfont=dict(size=10, color='#2f855a'),
            customdata=np.column_stack([pre_edge, war_edge]),
            hovertemplate='Family=%{y}<br>Pre-war weighted information=%{x:.4f}<br>Pre-war signed edge=%{customdata[0]:.4f}<br>War signed edge=%{customdata[1]:.4f}<extra></extra>',
            legendgroup='period',
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=war_info,
            y=labels,
            mode='markers+text',
            name='War',
            marker=dict(color='#c53030', size=12, symbol='diamond'),
            text=[f"{value:.4f}" for value in war_info],
            textposition='middle right',
            textfont=dict(size=10, color='#c53030'),
            customdata=np.column_stack([pre_edge, war_edge]),
            hovertemplate='Family=%{y}<br>War weighted information=%{x:.4f}<br>Pre-war signed edge=%{customdata[0]:.4f}<br>War signed edge=%{customdata[1]:.4f}<extra></extra>',
            legendgroup='period',
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )

fig.update_xaxes(title_text='Weight share x information magnitude', row=1, col=1)
fig.update_xaxes(title_text='Weight share x information magnitude', row=1, col=2)
fig.update_layout(
    title='Which Feature Families Actually Carry Weighted Information?',
    height=430,
    margin=dict(t=90, l=40, r=40, b=40),
    legend=dict(orientation='h', y=1.08, x=0.01),
)
fig.show()


The baseline draws most of its usable war information from `Macro`, `Volatility`, `Structure`, and `Returns`, while `Sent` adds war-relevant information in `Policy/Event`, `Agent`, and `Sentiment`. This is one of the clearest visual explanations for the regime split: the baseline remains a market-structure model, while `Sent` gains extra information from interpreting the war-related text layer.

### Where the Full Sent Model Shifts Weight Across Feature Families

After seeing which families carry usable information, this plot shows where the full `Sent` model reallocates capacity compared with the baseline. Positive bars mean `Sent` allocates more family weight than the baseline; negative bars mean it allocates less.

In [71]:
# Compare family-level weight share in the full Sent model versus the baseline backbone.
fig = px.bar(
    family_compare.sort_values('sent_minus_baseline'),
    x='sent_minus_baseline',
    y='family',
    orientation='h',
    title='Where the Full Sent Model Shifts Weight Across Feature Families',
    color='sent_minus_baseline',
    color_continuous_scale='RdBu',
    hover_data={'Baseline': ':.3%', 'Sent total': ':.3%', 'sent_minus_baseline': ':.3%'},
)
fig.update_layout(xaxis_title='Full sent minus baseline family weight share', yaxis_title='Feature family', height=560)
fig.show()


The full `Sent` model explicitly reallocates model capacity toward `Sentiment` and `Agent` families relative to the baseline. That matches the earlier feature-signal evidence and the top-weighted text inputs such as `supply_disruption_share`, `agent_company_exposure`, and `geopolitics_share`. The war-period gain is therefore structurally interpretable rather than just descriptive.

### How Much Accuracy Comes from Combining Features?

The final check is whether the result comes from one dominant variable or from combining multiple weak signals. The full-model bar should be compared with the best-single-feature bar in the same model and period. The gap is the value of feature combination and sequence structure.

In [72]:
# Benchmark the full models against the best single feature in each regime.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def best_single_feature_accuracy(frame: pd.DataFrame, features: list[str], period: str) -> tuple[str | None, float]:
    subset = frame[frame['period'] == period].copy()
    best_feature = None
    best_accuracy = np.nan
    for feature in features:
        if feature not in subset.columns:
            continue
        sample = subset[[feature, 'target']].replace([np.inf, -np.inf], np.nan).dropna()
        if len(sample) < 30 or sample['target'].nunique() < 2 or sample[feature].nunique() < 2:
            continue
        X = sample[[feature]].to_numpy()
        y = sample['target'].astype(int).to_numpy()
        min_class = int(np.bincount(y).min())
        n_splits = min(5, min_class)
        if n_splits < 2:
            continue
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        prob = np.zeros(len(y))
        for train_idx, test_idx in cv.split(X, y):
            model = LogisticRegression(max_iter=500)
            model.fit(X[train_idx], y[train_idx])
            prob[test_idx] = model.predict_proba(X[test_idx])[:, 1]
        pred = (prob >= 0.5).astype(int)
        accuracy = float((pred == y).mean())
        if pd.isna(best_accuracy) or accuracy > best_accuracy:
            best_accuracy = accuracy
            best_feature = feature
    return best_feature, best_accuracy

baseline_overlap = analysis[['date', 'ticker', 'target', 'period']].merge(
    quant_panel[['date', 'ticker'] + baseline_features],
    on=['date', 'ticker'],
    how='left',
)
sent_overlap = analysis[['date', 'ticker', 'target', 'period']].merge(
    news_panel[['date', 'ticker'] + sent_features],
    on=['date', 'ticker'],
    how='left',
)

rows = []
for model_name, feature_frame, feature_list, correct_col in [
    ('Baseline', baseline_overlap, baseline_features, 'baseline_correct'),
    ('Sent', sent_overlap, sent_features, 'sent_correct'),
]:
    for period in ['Pre-war', 'War']:
        best_feature, best_accuracy = best_single_feature_accuracy(feature_frame, feature_list, period)
        full_accuracy = float(behavior[behavior['period'] == period][correct_col].mean())
        rows.append(
            {
                'model': model_name,
                'period': period,
                'full_model_accuracy': full_accuracy,
                'best_single_accuracy': best_accuracy,
                'synergy_gain': full_accuracy - best_accuracy,
                'best_feature': best_feature,
            }
        )
synergy_df = pd.DataFrame(rows)

plot_df = synergy_df.melt(
    id_vars=['model', 'period', 'best_feature', 'synergy_gain'],
    value_vars=['best_single_accuracy', 'full_model_accuracy'],
    var_name='series',
    value_name='accuracy',
)
plot_df['series'] = plot_df['series'].map({
    'best_single_accuracy': 'Best single feature',
    'full_model_accuracy': 'Full model',
})

fig = px.bar(
    plot_df,
    x='period',
    y='accuracy',
    color='series',
    barmode='group',
    facet_col='model',
    title='How Much Accuracy Comes from Combining Features?',
    color_discrete_map={'Best single feature': '#a0aec0', 'Full model': '#1a202c'},
    hover_data={'best_feature': True, 'synergy_gain': ':.3f'},
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(height=560, xaxis_title='', yaxis_title='Accuracy')
fig.show()


Neither model is being driven by one magic input. The full models beat their best single features in both periods. In the war window, the best single `Sent` feature is `gpr_threat`, but the full `Sent` model still improves on it. That means the war edge is coming from feature combination, not from one isolated variable.

## Conclusion

The research question was whether text interpretation becomes more useful once the market moves into a geopolitical supply-risk regime. In this run, the answer is **yes**, but in a specific way. The baseline remains a strong model in both periods, which matters because this is not a weak-baseline story. Before the war, the two models are fairly close. During the war, however, `Baseline` stays roughly flat while `Sent` improves to **0.7079**, creating a much wider gap.

The rest of the notebook explains that gap rather than just reporting it. The agreement plot shows that the war-period outperformance is real incremental signal: `Sent` gets **14** war cases right that `Baseline` misses, while `Baseline` gets only **5** that `Sent` misses. The event-day plots then show where that edge appears. It is strongest on official event days and clusters around dated war-related information flow, which is exactly what the research idea would predict if language becomes more valuable when the market is reacting to shipping risk, sanctions, escalation, and supply disruption.

The feature and weighting sections explain why the models diverge. The baseline is still being supported by market and macro structure, but much of that information is inverse or unstable rather than cleanly directional. The `Sent` model, by contrast, benefits from war-salient text features such as `geopolitics_share`, `shipping_share`, and `supply_risk_intensity`, and the weighting plots show that the trained model actually reallocates capacity toward `Sentiment` and `Agent` families. The synergy plot then closes the loop: the gain is not coming from one lucky variable, but from combining several weak but complementary signals.

The practical conclusion is therefore narrow and defensible: a broad market-and-event baseline remains useful before and during the shock, but once oil pricing becomes driven by geopolitical supply narratives, a model that interprets those narratives adds real incremental predictive value.